# 📊 Chapter 1: Exploratory Data Analysis (EDA)
**Reference:** *Practical Statistics for Data Scientists (2nd Edition)* by Peter Bruce, Andrew Bruce, & Peter Gedeck

---

## 1.1 Introduction to EDA
Exploratory Data Analysis (EDA) was pioneered by John Tukey in 1977. It is the philosophy that data should be explored visually and statistically to discover structural patterns, anomalies, and relationships *before* formal statistical modeling or machine learning is applied.

Data science is fundamentally an iterative process of EDA -> Modeling -> More EDA. This chapter covers the foundational metrics and visualizations used to understand the shape, central tendency, dispersion, and correlations of data.

### Elements of Structured Data
- **Continuous (Numeric):** Data that can take on any value in an interval (e.g., wind speed, time).
- **Discrete (Numeric):** Data that can take on only integer values (e.g., count of events).
- **Categorical (String/Factor):** Data that can take on a specific set of values representing a set of possible categories (e.g., State, Color).
- **Binary (Boolean):** A special case of categorical data with just two values (e.g., 0/1, True/False).
- **Ordinal (Categorical):** Categorical data that has an explicit ordering (e.g., Rating from 1 to 5).

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for better visualizations
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')

print("Libraries successfully imported for Exploratory Data Analysis.")

## 1.2 Creating the Synthetic "State" Dataset
Throughout Chapter 1, the authors use a dataset containing Population and Murder Rates for US States. To make this notebook fully reproducible without external CSV files, we will generate a representative Pandas DataFrame.

In [ ]:
# Creating a synthetic dataset mirroring the book's 'state.csv'
state_data = {
    'State': ['Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut', 'Delaware', 'Florida', 'Georgia'],
    'Population': [4779736, 710231, 6392017, 2915918, 37253956, 5029196, 3574097, 897934, 18801310, 9687653],
    'Murder_Rate': [5.7, 5.6, 4.7, 5.6, 4.4, 2.8, 2.4, 5.8, 5.8, 5.7],
    'Abbreviation': ['AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA']
}

state_df = pd.DataFrame(state_data)
print("Sample of the Rectangular Data (DataFrame):")
display(state_df.head())

## 1.3 Estimates of Location (Central Tendency)
Variables with measured or count data might have thousands of distinct values. A basic step in exploring your data is getting a "typical value" for each feature.

- **Mean (Average):** The sum of all values divided by the number of values. Highly sensitive to extreme values (outliers).
- **Trimmed Mean:** The average of all values after dropping a fixed number of extreme values (e.g., dropping the top 10% and bottom 10%). This is a **robust** estimate.
- **Median:** The value such that one-half of the data lies above and below it. The median is inherently robust to outliers.

Let's compute these location estimates for the Population feature.

In [ ]:
# 1. Mean
mean_pop = state_df['Population'].mean()

# 2. Trimmed Mean (dropping 10% from each end)
trim_mean_pop = stats.trim_mean(state_df['Population'], proportiontocut=0.10)

# 3. Median
median_pop = state_df['Population'].median()

print(f"Mean Population        : {mean_pop:,.0f}")
print(f"Trimmed Mean (10%)     : {trim_mean_pop:,.0f}")
print(f"Median Population      : {median_pop:,.0f}")

print("\nObservation: The Mean is significantly higher than the Median. This indicates the data is right-skewed, heavily influenced by massive populations (like California).")

## 1.4 Estimates of Variability (Dispersion)
Location is only half the story. We also need to measure variability (also called dispersion), which measures whether the data values are tightly clustered or spread out.

- **Variance:** The sum of squared deviations from the mean, divided by $n-1$. 
- **Standard Deviation:** The square root of the variance. It is on the same scale as the original data.
- **Mean Absolute Deviation (MAD):** The mean of the absolute values of the deviations from the mean. (Note: The book refers to Median Absolute Deviation from the median as MAD, which is more robust).
- **Interquartile Range (IQR):** The difference between the 75th percentile and the 25th percentile. This is a highly robust measure of variability.

In [ ]:
# 1. Standard Deviation
std_pop = state_df['Population'].std()

# 2. Interquartile Range (IQR)
iqr_pop = state_df['Population'].quantile(0.75) - state_df['Population'].quantile(0.25)

# 3. Median Absolute Deviation from the Median (Robust MAD)
mad_pop = stats.median_abs_deviation(state_df['Population'])

print(f"Standard Deviation     : {std_pop:,.0f}")
print(f"Interquartile Range    : {iqr_pop:,.0f}")
print(f"Median Abs Deviation   : {mad_pop:,.0f}")

print("\nObservation: Like the mean, the standard deviation is inflated by outliers. The IQR and MAD provide a much more stable estimate of the 'typical' spread of the data.")

## 1.5 Exploring the Data Distribution
While estimates of location and variability boil data down to single numbers, looking at the entire distribution provides a holistic view.

### Percentiles and Boxplots
Percentiles tell us how data is distributed across its sorted range. A **Boxplot**, introduced by Tukey, is a quick visual summary of these percentiles. The box bounds the IQR (25th to 75th percentile), the line inside is the median, and the "whiskers" extend to the range of the data (excluding outliers).

In [ ]:
# Calculate Percentiles
percentiles = [0.05, 0.25, 0.5, 0.75, 0.95]
pop_percentiles = state_df['Population'].quantile(percentiles)
print("Population Percentiles:\n", pop_percentiles, "\n")

# Create a Boxplot
plt.figure(figsize=(8, 4))
sns.boxplot(x=state_df['Population'] / 1_000_000, color='lightblue')
plt.title('Boxplot of State Populations (in Millions)')
plt.xlabel('Population (Millions)')
plt.show()

print("The dot on the far right represents an outlier (California). The box shows where the bulk of the states fall.")

### Frequency Tables, Histograms, and Density Plots
- **Frequency Table:** Divides the variable range into equally spaced segments (bins) and counts how many values fall in each segment.
- **Histogram:** The visual representation of a frequency table. The x-axis shows the bins, and the y-axis shows the count.
- **Density Plot:** A smoothed version of a histogram (usually Kernel Density Estimation). The y-axis is scaled so the total area under the curve equals 1.

In [ ]:
# 1. Frequency Table (Using pd.cut to bin the data)
binned_pop = pd.cut(state_df['Population'], bins=5)
print("Frequency Table (Population Bins):\n", binned_pop.value_counts().sort_index(), "\n")

# 2. Histogram and Density Plot combined
plt.figure(figsize=(10, 5))

# We plot the histogram and the KDE (Kernel Density Estimate) curve together
sns.histplot(state_df['Population'] / 1_000_000, bins=5, kde=True, stat="density", color='skyblue', edgecolor='black')

plt.title('Histogram and Density Plot of State Populations')
plt.xlabel('Population (Millions)')
plt.ylabel('Density')
plt.show()

## 1.6 Exploring Binary and Categorical Data
For categorical data, simple proportions or percentages tell the story.

- **Mode:** The most commonly occurring category.
- **Expected Value:** A special calculation used when categories represent discrete numerical outcomes (like a lottery or expected revenue). It is calculated by multiplying each outcome by its probability and summing them: $EV = \sum P(x_i) \times x_i$.
- **Bar Charts:** The primary visual tool for categorical data. (Note: Bar charts are NOT histograms. Bar charts have spaces between bars representing distinct categories; histograms represent a continuous scale).

In [ ]:
# Simulating Categorical Data: Telecom churn causes
churn_causes = pd.Series(['Price', 'Service', 'Price', 'Price', 'Network', 'Service', 'Price', 'Network'])

print("Mode of Churn Causes:", churn_causes.mode()[0])

# Plotting a Bar Chart
plt.figure(figsize=(7, 4))
churn_counts = churn_causes.value_counts()
sns.barplot(x=churn_counts.index, y=churn_counts.values, palette='viridis')
plt.title('Bar Chart of Telecom Churn Causes')
plt.ylabel('Frequency')
plt.show()

# Expected Value Example: Expected revenue of an ad campaign
outcomes = np.array([0, 10, 50])      # Revenue outcomes ($)
probabilities = np.array([0.8, 0.15, 0.05]) # Probabilities

expected_value = np.sum(outcomes * probabilities)
print(f"Expected Value of Ad Click: ${expected_value:.2f}")

## 1.7 Correlation
Exploratory Data Analysis in many modeling projects involves examining the relationship between predictors, and between predictors and the target variable.

The **Pearson Correlation Coefficient (r)** measures the extent to which numeric variables are linearly related. 
- $r = +1$: Perfect positive correlation
- $r = -1$: Perfect negative correlation
- $r = 0$: No linear correlation

Variables are positively correlated if high values of $X$ go with high values of $Y$. They are negatively correlated if high values of $X$ go with low values of $Y$.

To visualize correlation, we use **Scatter Plots** and **Correlation Matrices (Heatmaps)**.

In [ ]:
# Create synthetic data for multiple variables to show a Correlation Matrix
np.random.seed(42)
n = 100
tech_stocks = pd.DataFrame({
    'AAPL': np.random.normal(0, 1, n),
    'MSFT': np.random.normal(0, 1, n),
})
# Inject correlation
tech_stocks['MSFT'] = tech_stocks['AAPL'] * 0.7 + np.random.normal(0, 0.5, n)
tech_stocks['GOOG'] = tech_stocks['AAPL'] * 0.4 + np.random.normal(0, 0.8, n)
tech_stocks['GOLD'] = np.random.normal(0, 1, n) # Uncorrelated asset

# 1. Calculate the Correlation Matrix
corr_matrix = tech_stocks.corr()
print("Pearson Correlation Matrix:\n", np.round(corr_matrix, 2))

# 2. Plotting a Correlation Heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f")
plt.title('Correlation Heatmap')
plt.show()

# 3. Scatter Plot for highly correlated variables
plt.figure(figsize=(6, 5))
sns.scatterplot(x='AAPL', y='MSFT', data=tech_stocks, alpha=0.7)
plt.title('Scatter Plot: AAPL vs MSFT Returns')
plt.xlabel('AAPL Returns')
plt.ylabel('MSFT Returns')
plt.show()

## 1.8 Exploring Two or More Variables (Multivariate Analysis)
When datasets have hundreds of thousands of records, standard scatterplots become a dense, unreadable blob of points. We need tools to visualize density and relationships in large datasets.

### Hexagonal Binning and Contours (Numeric vs Numeric)
Instead of plotting points, we group the data into hexagonal bins and color the bin based on the number of records it contains. Alternatively, contour plots act like topographical maps of density.

### Boxplots and Violin Plots (Categorical vs Numeric)
To see how a numeric variable varies across different categories, side-by-side boxplots or violin plots are ideal. A violin plot is an enhancement of a boxplot that shows the density estimate on the y-axis.

In [ ]:
# Generate Large Dataset for Hexbin
large_x = np.random.normal(0, 1, 10000)
large_y = large_x * 0.5 + np.random.normal(0, 1, 10000)
large_df = pd.DataFrame({'SqFt': large_x, 'TaxValue': large_y})

# 1. Hexagonal Binning Plot
plt.figure(figsize=(7, 5))
plt.hexbin(large_df['SqFt'], large_df['TaxValue'], gridsize=30, cmap='Blues')
cb = plt.colorbar(label='Count in Bin')
plt.title('Hexagonal Binning (Large Datasets)')
plt.xlabel('Square Feet (Standardized)')
plt.ylabel('Tax Value (Standardized)')
plt.show()

# 2. Violin Plot (Categorical vs Numeric)
# Simulating airline delay data
airline_data = pd.DataFrame({
    'Airline': ['Delta']*200 + ['United']*200 + ['Southwest']*200,
    'Delay_Minutes': np.concatenate([
        np.random.normal(10, 15, 200), 
        np.random.normal(20, 20, 200),
        np.random.normal(5, 10, 200)
    ])
})

plt.figure(figsize=(8, 5))
sns.violinplot(x='Airline', y='Delay_Minutes', data=airline_data, palette='Set2')
plt.title('Violin Plot of Flight Delays by Airline')
plt.xlabel('Airline')
plt.ylabel('Delay (Minutes)')
plt.axhline(0, color='gray', linestyle='--')
plt.show()

### Conclusion of Chapter 1
Exploratory Data Analysis is the critical first step in any data science project. With estimates of location and variability, you summarize the data mathematically. With histograms, boxplots, hexbins, and heatmaps, you summarize it visually.

These tools prevent you from feeding garbage into machine learning models and help uncover the true narrative hidden within the numbers.